# Module 3 (Part 2): Power Flow and Locational Marginal Pricing
## REE 4301 / IE 5300 - Energy Systems Modeling

In the transport companion you moved gas through pipelines. You chose the flow on every route, and the only limits were capacities. That is a **transport model**, and it is the right model for pipelines, railways and shipping.

**Electricity does not work that way.** Nobody chooses how power flows. It divides itself among every available path according to the physics of the network, and the grid operator's only levers are which generators run and what the network is built like.

This notebook does the same thing twice, as always:

1. **In gurobipy**, where you write the power-flow equations yourself and pull the prices out of the LP by hand.
2. **In PyPSA**, where you say `Line` instead of `Link` and all of it happens for you.

The point to carry out of here is one sentence: **a locational marginal price is the dual variable of a nodal balance constraint.** Not an economic theory bolted on afterwards - a number that falls out of the same LP you have been writing since Module 0.


In [ ]:
!pip install -q pypsa highspy gurobipy


In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import pypsa
import warnings
warnings.filterwarnings('ignore')

SOLVER = 'gurobi'
# SOLVER = 'highs'     # <-- uncomment: open source, no licence, no size cap

# For homework: paste your academic Web License Service key here.
WLS = {}   # {'WLSACCESSID': '...', 'WLSSECRET': '...', 'LICENSEID': 000000}
ENV = gp.Env(params=WLS) if (SOLVER == 'gurobi' and WLS) else None

print(f'solver: {SOLVER}'
      + ('  (academic WLS licence)' if ENV else '  (default licence)'))


---
# Part A - The system, and why it needs angles

Three buses in a triangle. Every line has the same reactance, *X* = 0.1.

| bus | what is there | capacity | cost |
|---|---|---|---|
| West Texas | wind | 150 MW | $0/MWh |
| Houston | gas | 100 MW | $40/MWh |
| Dallas | 120 MW of demand | - | - |

**The West Texas to Dallas line can carry only 70 MW.** The other two are effectively unlimited.


### First, predict

Wind is free and there is 150 MW of it. Demand is 120 MW. So run 120 MW of wind and pay nothing - except the direct line to Dallas only holds 70.

**Write down your answers before you run anything:**

1. How much wind and how much gas get dispatched?
2. What is the price of electricity at Dallas?

Most people answer 70 wind / 50 gas for the first, and $40 for the second. Both are wrong, and the reasons they are wrong are the entire content of this notebook.


### The DC power flow equations

A transport model would let you choose each line's flow. Here you cannot. Instead, each bus gets a **voltage angle** *θ*, and the flow on a line is fixed by the angle difference across it:

&nbsp;&nbsp;&nbsp;&nbsp;*F<sub>ij</sub>* = ( *θ<sub>i</sub>* − *θ<sub>j</sub>* ) / *X<sub>ij</sub>*

That single equation is what makes this a power-flow model. The optimiser picks generation; the angles then *determine* every flow at once. You get flow on lines you never asked to use.

Two consequences worth naming now:

- **Only angle differences matter.** Add 5 to every angle and nothing changes. So one bus must be pinned - the **reference** or **slack** bus - or the LP has infinitely many equally good solutions.
- **You cannot route power.** If you want less flow on a path, your only options are to change what generates where, or to change the network's reactances. Part E does the second.


---
# Part B - By hand, in gurobipy

The same five parts as every other model in this course, numbered the same way as the transport companion.

Watch for one thing in Part 5: we **keep** the three nodal balance constraint objects in named variables. Usually you would not bother. Here they are the whole point, because their dual variables are the prices.


In [ ]:
# ==========================================
# 1. Sets and Parameters (Data)
# ==========================================
buses = ['West Texas', 'Dallas', 'Houston']

demand = {'Dallas': 120.0}                        # D_j,  MW
gen_costs = {'WTX_Wind': 0.0, 'HOU_Gas': 40.0}    # c_g,  $/MWh
gen_max = {'WTX_Wind': 150.0, 'HOU_Gas': 100.0}   # Pbar_g,  MW

line_X = {'WTX_DAL': 0.1, 'DAL_HOU': 0.1, 'HOU_WTX': 0.1}   # X_ij, p.u.
line_limit = {'WTX_DAL': 70.0, 'DAL_HOU': 1000.0,
              'HOU_WTX': 1000.0}                  # Fbar_ij,  MW

# ==========================================
# 2. Model Initialization
# ==========================================
m = gp.Model('3_Bus_DCOPF', env=ENV) if ENV else gp.Model('3_Bus_DCOPF')
m.Params.OutputFlag = 0


In [ ]:
# ==========================================
# 3. Decision Variables
# ==========================================
# Generator dispatch (MW)
P_wind = m.addVar(lb=0, ub=gen_max['WTX_Wind'], name='P_wind')
P_gas = m.addVar(lb=0, ub=gen_max['HOU_Gas'], name='P_gas')

# Voltage angles (radians). These are decision variables too, even though
# nobody 'decides' them physically - the LP solves for the angles that
# make the flows consistent with the injections.
theta_wtx = m.addVar(lb=-GRB.INFINITY, name='theta_wtx')
theta_dal = m.addVar(lb=-GRB.INFINITY, name='theta_dal')
theta_hou = m.addVar(lb=-GRB.INFINITY, name='theta_hou')

# REFERENCE BUS: pin one angle to zero.
# DCOPF determines only angle DIFFERENCES, so without this the model has
# infinitely many optimal solutions and the solver may return any of them.
m.addConstr(theta_wtx == 0, 'Ref_Bus')


In [ ]:
# ==========================================
# 4. Objective Function
# ==========================================
m.setObjective(
    gen_costs['WTX_Wind'] * P_wind + gen_costs['HOU_Gas'] * P_gas,
    GRB.MINIMIZE)


In [ ]:
# ==========================================
# 5. Constraints
# ==========================================
# --- the DC power flow equation, F_ij = (theta_i - theta_j) / X_ij ---
# These are linear EXPRESSIONS, not new variables: each one is already
# determined by the angles.
Flow_WTX_DAL = (theta_wtx - theta_dal) / line_X['WTX_DAL']
Flow_DAL_HOU = (theta_dal - theta_hou) / line_X['DAL_HOU']
Flow_HOU_WTX = (theta_hou - theta_wtx) / line_X['HOU_WTX']

# --- thermal limit, in BOTH directions ---
m.addConstr(Flow_WTX_DAL <= line_limit['WTX_DAL'], 'Limit_WTX_DAL_Pos')
m.addConstr(Flow_WTX_DAL >= -line_limit['WTX_DAL'], 'Limit_WTX_DAL_Neg')

# --- nodal power balance at every bus (Kirchhoff's current law) ---
# generation - demand = net flow leaving the bus
#
# *** KEEP THESE OBJECTS. Their dual variables are the LMPs. ***
bal_wtx = m.addConstr(
    P_wind - 0 == Flow_WTX_DAL - Flow_HOU_WTX, 'Balance_WTX')
bal_dal = m.addConstr(
    0 - demand['Dallas'] == Flow_DAL_HOU - Flow_WTX_DAL, 'Balance_DAL')
bal_hou = m.addConstr(
    P_gas - 0 == Flow_HOU_WTX - Flow_DAL_HOU, 'Balance_HOU')


In [ ]:
# ==========================================
# 6. Optimize and Output
# ==========================================
m.optimize()
assert m.Status == GRB.OPTIMAL, f'solver status {m.Status}'

f_wd = Flow_WTX_DAL.getValue()
f_dh = Flow_DAL_HOU.getValue()
f_hw = Flow_HOU_WTX.getValue()

# Dallas's balance was written with demand on the LEFT, so its dual comes
# out with the opposite sign to the other two. Negate it to get the price.
lmp_wtx, lmp_dal, lmp_hou = bal_wtx.Pi, -bal_dal.Pi, bal_hou.Pi

print('dispatch')
print(f'  wind (West Texas) {P_wind.X:6.1f} MW')
print(f'  gas  (Houston)    {P_gas.X:6.1f} MW')
print()
print('line flows  (positive = in the direction named)')
print(f'  West Texas -> Dallas  {f_wd:6.1f} MW   (limit 70)')
print(f'  Dallas -> Houston     {f_dh:6.1f} MW')
print(f'  Houston -> West Texas {f_hw:6.1f} MW')
print()
print('locational marginal prices  (dual of each nodal balance)')
print(f'  West Texas ${lmp_wtx:6.2f} /MWh')
print(f'  Dallas     ${lmp_dal:6.2f} /MWh')
print(f'  Houston    ${lmp_hou:6.2f} /MWh')
print()
print(f'total generation cost ${m.ObjVal:,.2f} /hr')


---
# Part C - Read that output again

Three things happened that are worth more than the rest of the notebook.

### 1. The dispatch is not 70 / 50

The direct line is full at 70 MW, but **90 MW of wind is running**, not 70. The extra 20 MW leaves West Texas heading for *Houston*, and reaches Dallas the long way round. Nobody routed it there. The angles did.

That is **loop flow**, and it is the thing a transport model cannot represent. Power on a mesh network uses every path between two points, in proportion to how easy each path is.

### 2. Dallas costs $80/MWh, and nothing in the system costs $80

The most expensive generator running is gas at **$40**. Dallas's price is **twice that**. Before reading on: where can an $80 price come from when nothing in the system costs $80?

Ask what it actually costs to deliver one more MW to Dallas. You cannot just send more wind - the direct line is full. Pushing one more MW down that path means the angles have to shift, which pulls flow around the loop as well, and the only way to keep every bus balanced is to **turn gas up by more than one MW while turning wind down**. The system pays $40 twice over to move one MW. Hence $80.

This is why prices at a constrained location can exceed every generator's cost - and, in real markets, why they can go **negative** at a location that is exporting into a constraint.

### 3. The prices came out of the constraints, not out of an economic model

`bal_dal.Pi` is a *dual variable*. You did not write a pricing rule. You wrote a cost-minimising LP with a balance constraint at each bus, and the prices were already in there.


In [ ]:
# congestion rent: what the operator collects from the price difference
# across a line.  rent = flow x (price at the receiving end
#                              - price at the sending end)
rent_wd = f_wd * (lmp_dal - lmp_wtx)
rent_dh = f_dh * (lmp_hou - lmp_dal)
rent_hw = f_hw * (lmp_wtx - lmp_hou)

print(f'  West Texas -> Dallas  ${rent_wd:9,.2f} /hr')
print(f'  Dallas -> Houston     ${rent_dh:9,.2f} /hr')
print(f'  Houston -> West Texas ${rent_hw:9,.2f} /hr')
print(f'  {"total":21s} ${rent_wd + rent_dh + rent_hw:9,.2f} /hr')

print()
print('Consumers at Dallas pay  ${:,.2f}/hr'.format(120 * lmp_dal))
print('Generators are paid      ${:,.2f}/hr'.format(
      P_wind.X * lmp_wtx + P_gas.X * lmp_hou))
print('The difference is the congestion rent above: it does not go to'
      ' anyone who produced or consumed electricity.')


> **Exercise C.1.** The congestion rent is collected by the grid operator, not by any generator or consumer. In ERCOT it is paid out to the holders of Congestion Revenue Rights. Who *should* receive it, and what would happen to transmission investment if the answer were 'whoever built the line'?
>
> **Exercise C.2.** Raise the limit from 70 to 80 and re-run. Wind covers all 120 MW, gas shuts off - and every LMP goes to $0. Explain why the congestion rent vanishes at the same moment.


---
# Part D - The same model in PyPSA, one component at a time

Everything you just wrote by hand - the angle variables, the reference bus, the flow equations, the thermal limits, the nodal balances - is written for you by **one word**: `Line` instead of `Link`.

A `Link` is a controllable transport route. You set its flow. A `Line` is AC transmission, and its flow is decided by physics. That is the entire difference, and it is worth more than any other single fact in this module.


### Step 1: Initialize the network


In [ ]:
n = pypsa.Network()

# one snapshot: this is a single-hour problem, exactly like the gurobi one
n.set_snapshots([0])
print(n)


### Step 2: Add the buses

A **Bus** is a place where power balances. Each one becomes exactly the `Balance_*` constraint you wrote by hand in Part B - and each one gets an angle variable and a dual variable, for free.


In [ ]:
for b in ['West Texas', 'Dallas', 'Houston']:
    n.add('Bus', b)

print(list(n.buses.index))


### Step 3: Add the generators

`p_nom` is the capacity and `marginal_cost` is the running cost. Note `p_nom`, not `p_nom_extendable` - this is a **dispatch** problem. Nothing is being built; we are only deciding what to run.


In [ ]:
# Generator.p_nom         maps to the capacity limit Pbar_g
# Generator.marginal_cost maps to the cost coefficient c_g
n.add('Generator', 'WTX_Wind', bus='West Texas',
      p_nom=150, marginal_cost=0)
n.add('Generator', 'HOU_Gas', bus='Houston',
      p_nom=100, marginal_cost=40)

print(n.generators[['bus', 'p_nom', 'marginal_cost']].to_string())


### Step 4: Add the demand

`p_set` is a fixed requirement - the *D* on the right-hand side of the Dallas balance constraint.


In [ ]:
# Load.p_set maps to the demand requirement D_j
n.add('Load', 'Dallas_Demand', bus='Dallas', p_set=120)

print(n.loads[['bus', 'p_set']].to_string())


### Step 5: Add the lines - the step that matters

Two arguments, and they are not interchangeable with anything you used in the transport notebook:

- **`x`** is the reactance. This is what invokes the flow equation. Give PyPSA an `x` and it will create the angle variables, write *F* = (*θ<sub>i</sub>* − *θ<sub>j</sub>*)/*X* for every line, and pin a reference bus. There is no argument for 'please apply Kirchhoff' - `x` *is* that argument.
- **`s_nom`** is the thermal rating, enforced in both directions, exactly like the two `Limit_WTX_DAL_*` constraints you wrote by hand.


In [ ]:
# Line.x     maps to the reactance X_ij in F_ij = (theta_i - theta_j)/X_ij
# Line.s_nom maps to the thermal limit Fbar_ij (enforced +/-)
n.add('Line', 'WTX_DAL', bus0='West Texas', bus1='Dallas',
      x=0.1, s_nom=70)          # <-- the bottleneck
n.add('Line', 'DAL_HOU', bus0='Dallas', bus1='Houston',
      x=0.1, s_nom=1000)
n.add('Line', 'HOU_WTX', bus0='Houston', bus1='West Texas',
      x=0.1, s_nom=1000)

print(n.lines[['bus0', 'bus1', 'x', 's_nom']].to_string())


### Step 6: Look at what PyPSA is about to solve

Before solving, ask it what it wrote. Compare this list against the constraints you typed in Part B.


In [ ]:
lp = n.optimize.create_model()

print('variables:  ', list(lp.variables))
print()
print('constraints:', list(lp.constraints))


`Bus-nodal_balance` is your three `Balance_*` constraints. `Line-fix-s_nom-upper` and `-lower` are your two thermal limits. And `Kirchhoff-Voltage-Law` is the one you had to derive angles for.

Notice what PyPSA did *not* need: an explicit angle variable per bus. For a network this small it solves an equivalent formulation over the network's independent loops - one KVL constraint per loop instead of one angle per bus. Same physics, same answer, fewer variables. Another reminder that the tool is doing bookkeeping you could do yourself, but would rather not.


### Step 7: Solve, and read the prices off the buses


In [ ]:
n.optimize(solver_name=SOLVER, env=ENV)

out = pd.DataFrame({
    'dispatch MW': n.generators_t.p.iloc[0],
})
print(out.to_string())
print()
print('line flows (MW, positive = bus0 -> bus1)')
print(n.lines_t.p0.iloc[0].round(1).to_string())
print()
print('locational marginal prices ($/MWh)')
print(n.buses_t.marginal_price.iloc[0].round(2).to_string())
print()
print(f'total generation cost ${n.objective:,.2f} /hr')


### The check that makes the point

`bal_dal.Pi` in your hand-written Gurobi model and `n.buses_t.marginal_price['Dallas']` in PyPSA are **the same number**, because they are the same dual variable of the same constraint.


In [ ]:
py_lmp = n.buses_t.marginal_price.iloc[0]
py_gen = n.generators_t.p.iloc[0]

rows = [
    ('wind MW', P_wind.X, py_gen['WTX_Wind']),
    ('gas MW', P_gas.X, py_gen['HOU_Gas']),
    ('LMP West Texas', lmp_wtx, py_lmp['West Texas']),
    ('LMP Dallas', lmp_dal, py_lmp['Dallas']),
    ('LMP Houston', lmp_hou, py_lmp['Houston']),
    ('total cost', m.ObjVal, n.objective),
]

print(f'{"quantity":16s} {"gurobipy":>12s} {"PyPSA":>12s}  match')
for label, a, b in rows:
    ok = 'yes' if abs(a - b) < 1e-6 else 'NO'
    print(f'{label:16s} {a:12.2f} {b:12.2f}  {ok}')

assert all(abs(a - b) < 1e-6 for _, a, b in rows), 'the two models differ'
print()
print('Same model. PyPSA did not invent new mathematics -'
      ' it wrote your constraints for you.')


---
# Part E - Where does the loop flow go?

You saw 20 MW take the long way round at a 70 MW limit. That number is not a coincidence - for this network you can work it out in closed form. With all three reactances equal, the flow on Houston to West Texas is exactly

&nbsp;&nbsp;&nbsp;&nbsp;*F<sub>HW</sub>* = 120 − 2*L*

where *L* is the limit on the West Texas to Dallas line.

**Predict before running:** at what limit does the loop flow disappear entirely? And what happens above that?

One of the five rows below will not solve at all. Before you run it, work out which one and why - it is an arithmetic question about the gas plant, not about the network.


In [ ]:
rows = []
for limit in [40, 50, 60, 70, 80]:
    k = pypsa.Network()
    k.set_snapshots([0])
    for b in ['West Texas', 'Dallas', 'Houston']:
        k.add('Bus', b)
    k.add('Generator', 'WTX_Wind', bus='West Texas',
          p_nom=150, marginal_cost=0)
    k.add('Generator', 'HOU_Gas', bus='Houston',
          p_nom=100, marginal_cost=40)
    k.add('Load', 'Dallas_Demand', bus='Dallas', p_set=120)
    k.add('Line', 'WTX_DAL', bus0='West Texas', bus1='Dallas',
          x=0.1, s_nom=limit)
    k.add('Line', 'DAL_HOU', bus0='Dallas', bus1='Houston',
          x=0.1, s_nom=1000)
    k.add('Line', 'HOU_WTX', bus0='Houston', bus1='West Texas',
          x=0.1, s_nom=1000)
    status, condition = k.optimize(solver_name=SOLVER, env=ENV)

    if condition != 'optimal':
        # not a bug - see the note below the table
        rows.append({'limit MW': limit, 'wind MW': None, 'gas MW': None,
                     'HOU->WTX MW': None, 'LMP Dallas': None,
                     'cost $/hr': None, 'status': 'CANNOT SOLVE'})
        print(f'  limit {limit} MW: solver said {condition!r}')
        continue

    g = k.generators_t.p.iloc[0]
    f = k.lines_t.p0.iloc[0]
    p = k.buses_t.marginal_price.iloc[0]
    rows.append({'limit MW': limit,
                 'wind MW': round(g['WTX_Wind'], 1),
                 'gas MW': round(g['HOU_Gas'], 1),
                 'HOU->WTX MW': round(f['HOU_WTX'], 1),
                 'LMP Dallas': round(p['Dallas'], 2),
                 'cost $/hr': round(k.objective, 2),
                 'status': 'optimal'})

sweep = pd.DataFrame(rows).set_index('limit MW')
print(sweep.to_string())


> **The 40 MW row is infeasible, and that is the correct answer.** With only 40 MW able to reach Dallas directly, the rest has to come from gas - but the formula says gas would need 240 − 3*L* = 120 MW, and the plant is only 100 MW. The model refuses. Below **L = 46.7 MW this network cannot serve Dallas at all**, and no amount of wind in West Texas changes that.
>
> An infeasible model is information, not a failure. It told you the system's breaking point without your having to search for it.
>
> **Now read the `HOU->WTX` column against the formula.** It is +20, 0, −20, −40 as the limit goes 50, 60, 70, 80 - exactly 120 − 2*L*.
>
> **At a 60 MW limit the loop flow is exactly zero** - the one setting where this network happens to behave like a simple transport model. Sitting either side of it, power flows through Houston in *opposite directions*: below 60 the gas plant is helping supply West Texas, above 60 the wind farm is exporting through Houston.
>
> **Exercise E.1.** At a 60 MW limit, a transport model and a power-flow model give the same answer. Explain why that is a coincidence and not a reassurance. What would you have concluded about this network if 60 MW were the only case you had ever tested?
>
> **Exercise E.2.** The Dallas LMP column changes with the limit. At which limit does congestion pricing switch off, and what is the price then?


---
# Part F - Reactance steers power

Everything so far used *X* = 0.1 on all three lines. Now change **only** the reactance of the Houston to West Texas line - the loop path - and leave every capacity exactly as it was.

**Predict first.** The direct West Texas to Dallas line is already full at 70 MW and stays full in every case below. So making the loop path easier or harder cannot move power onto the direct route. What does it change?

**This is the lever a transmission planner actually has.** You cannot tell power where to go, but you can change how hard each path is - by building a parallel line, by choosing a conductor, by switching a series device in or out.


In [ ]:
rows = []
for label, x_hw in [('easier loop  X=0.05', 0.05),
                    ('base         X=0.10', 0.10),
                    ('harder loop  X=0.20', 0.20)]:
    k = pypsa.Network()
    k.set_snapshots([0])
    for b in ['West Texas', 'Dallas', 'Houston']:
        k.add('Bus', b)
    k.add('Generator', 'WTX_Wind', bus='West Texas',
          p_nom=150, marginal_cost=0)
    k.add('Generator', 'HOU_Gas', bus='Houston',
          p_nom=100, marginal_cost=40)
    k.add('Load', 'Dallas_Demand', bus='Dallas', p_set=120)
    k.add('Line', 'WTX_DAL', bus0='West Texas', bus1='Dallas',
          x=0.1, s_nom=70)
    k.add('Line', 'DAL_HOU', bus0='Dallas', bus1='Houston',
          x=0.1, s_nom=1000)
    k.add('Line', 'HOU_WTX', bus0='Houston', bus1='West Texas',
          x=x_hw, s_nom=1000)
    k.optimize(solver_name=SOLVER, env=ENV)

    g = k.generators_t.p.iloc[0]
    f = k.lines_t.p0.iloc[0]
    p = k.buses_t.marginal_price.iloc[0]
    rows.append({'scenario': label,
                 'wind MW': round(g['WTX_Wind'], 1),
                 'gas MW': round(g['HOU_Gas'], 1),
                 'WTX->DAL MW': round(f['WTX_DAL'], 1),
                 'HOU->WTX MW': round(f['HOU_WTX'], 1),
                 'LMP Dallas': round(p['Dallas'], 2),
                 'cost $/hr': round(k.objective, 2)})

print(pd.DataFrame(rows).set_index('scenario').T.to_string())


### Read that table carefully - it contains a trap

**`WTX->DAL` is 70 MW in all three columns.** The direct line is full regardless. So what actually changed is the *loop*: 40, 20, then 10 MW. Easier loop path, more wind reaches Dallas the long way - 110 MW, 90 MW, 80 MW - and gas covers whatever wind cannot deliver.

**Total cost goes $400, $1,200, $1,600.** Nobody built or removed a single MW of capacity. The only thing that changed was how hard one path is.

Now the trap. Look at the Dallas LMP: **$120, $80, $60.** The price is *highest* in the cheapest system and *lowest* in the most expensive one. That is not an error.

For this network the relationship is exact:

&nbsp;&nbsp;&nbsp;&nbsp;LMP<sub>Dallas</sub> = LMP<sub>Houston</sub> × ( 1 + *X*<sub>WD</sub> / *X*<sub>HW</sub> )

Check it: 40 × (1 + 0.1/0.05) = 120. 40 × (1 + 0.1/0.1) = 80. 40 × (1 + 0.1/0.2) = 60.

The easier the loop path, the more leverage one extra MW at Dallas has over the whole system - so the *marginal* price climbs even as the *total* bill falls. **Total cost and marginal price answer different questions.** Confusing the two is among the most common and most expensive mistakes in energy analysis: a location with a high LMP is not necessarily a location where the system is doing badly.

> **Exercise F.1.** You are a planner trying to relieve congestion on the West Texas to Dallas line, and you have budget for exactly one project. Would you (a) build a second parallel West Texas to Dallas line, or (b) upgrade the Houston to West Texas line to lower its reactance? Use the table to argue for one, then test the other by editing the cell above.
>
> **Exercise F.2.** A generator developer is choosing where to site a new plant and picks Dallas because its LMP is the highest on the system. Using the X=0.05 column, explain why that reasoning is dangerous.
>
> **Exercise F.3 - the one worth putting in a report.** A new data centre opens at Dallas and demand rises from 120 MW to 200 MW. Is the system still feasible with the 70 MW limit? Find the largest Dallas demand this network can serve, and say which constraint stops it.


---
*Before class: the price at the constrained bus rose above every generator's cost. If you were siting a large load, would you rather sit there or at the cheap bus - and what would your arrival do to the price you picked it for?*

### Sources
- DC power-flow approximation and the reference-bus requirement: any power-systems text; see also the PyPSA docs on `Line` and `Kirchhoff-Voltage-Law`.
- ERCOT nodal prices and Congestion Revenue Rights: ercot.com market information.
- The $22.14/MWh delivered gas figure used elsewhere in this course is built in the SB6 notebook; the $40/MWh here is a round number chosen to make the arithmetic checkable by hand.
